# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fkashaf19-afk/ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

**Research question.** Can page-level search signals (recent impressions, clicks, CTR, average position, and static content properties) rank which pages will lose more than 20% of their impressions in the next 30 days?

**Decision it supports.** A content team cannot review every page each month. This work orders pages so an editor knows which to open first.

**Unit, output, action.** Unit: one page. Output: a risk rank with descriptive reason codes and a priority tier. Action: a human reviews the page. A wrong call costs editor time, so the list orders attention and never triggers automatic changes.

**Lane:** Refresh / Content Opportunity Scoring.

## 2. Data

**Release:** FlyRank ML Internship warehouse (Hugging Face, gated), build v20260703. **Tables used:** `dim_clients`, `dim_content`, `fact_content_daily_performance` (2025-01-27 to 2026-06-30). The query-level table was not used.

**Filters:** Search Console rows only (`gsc_data_available IS TRUE`). Clients with no search history, or with history starting less than 60 days before a cutoff, were excluded. A page is kept when it has at least 25 days of data in each of its windows and at least 30 impressions in the 30 days before the cutoff.

**Left out on purpose:** export-time fields (`is_deleted`, `is_published`, `backlinks`, `category_count`, `last_optimized_date`) that could reflect events after a cutoff, and all identifiers as features. Missing values are kept as missing, not filled with 0. Nothing here names a client, URL or query.

*The two cells below check access, then confirm table sizes, grain and the three-valued availability flags before any modelling.*

In [20]:
!pip -q install duckdb huggingface_hub

import os
from collections import Counter
from google.colab import userdata
from huggingface_hub import HfApi

# token stays in Colab Secrets, never pasted in the notebook
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

REPO = "FlyRank/internship-warehouse"
api = HfApi(token=os.environ["HF_TOKEN"])

files = api.list_repo_files(REPO, repo_type="dataset")
print("Total files in repo:", len(files))

# How many files per top-level folder / table
top = Counter(f.split("/")[0] for f in files)
print("\nFiles per top-level item:")
for k, v in top.items():
    print(f"  {k}: {v}")

# Show a few example paths per table so we can write exact queries
print("\nExample paths:")
seen = Counter()
for f in files:
    key = f.split("/")[0]
    if seen[key] < 3:
        print("  ", f)
        seen[key] += 1

Total files in repo: 24

Files per top-level item:
  .gitattributes: 1
  README.md: 1
  dim_clients.parquet: 1
  dim_content.parquet: 1
  fact_content_daily_performance: 18
  fact_content_daily_performance_sample.parquet: 1
  fact_content_query_90d.parquet: 1

Example paths:
   .gitattributes
   README.md
   dim_clients.parquet
   dim_content.parquet
   fact_content_daily_performance/month=2025-01/data_0.parquet
   fact_content_daily_performance/month=2025-02/data_0.parquet
   fact_content_daily_performance/month=2025-03/data_0.parquet
   fact_content_daily_performance_sample.parquet
   fact_content_query_90d.parquet


In [10]:
import os, duckdb
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")

BASE = "hf://datasets/FlyRank/internship-warehouse"
CLIENTS = f"{BASE}/dim_clients.parquet"
CONTENT = f"{BASE}/dim_content.parquet"
MONTH   = f"{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet"

def show(sql, title=None):
    if title: print(f"\n=== {title} ===")
    print(con.execute(sql).df().to_string(index=False))

# 3. One mid-panel month: size, date range
show(f"""
SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
       COUNT(DISTINCT content_hash_id) AS contents, COUNT(DISTINCT client_hash_id) AS clients
FROM '{MONTH}'
""", "month=2026-03 summary")

# Grain probe (should be empty)
show(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
FROM '{MONTH}' GROUP BY 1,2,3 HAVING COUNT(*) > 1 LIMIT 5
""", "Grain probe (should be empty)")

# Three-valued flags in this month (IS TRUE / IS FALSE / NULL)
show(f"""
SELECT
  COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)  AS gsc_true,
  COUNT(*) FILTER (WHERE gsc_data_available IS FALSE) AS gsc_false,
  COUNT(*) FILTER (WHERE gsc_data_available IS NULL)  AS gsc_null,
  COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS ga4_true,
  COUNT(*) FILTER (WHERE ga4_data_available IS FALSE) AS ga4_false,
  COUNT(*) FILTER (WHERE ga4_data_available IS NULL)  AS ga4_null
FROM '{MONTH}'
""", "Availability flags (month=2026-03)")

# 4. Client history coverage
show(f"""
SELECT COUNT(*) AS clients,
       MIN(gsc_data_start) AS earliest_gsc, MAX(gsc_data_start) AS latest_gsc,
       COUNT(*) FILTER (WHERE gsc_data_start IS NULL) AS null_gsc_start,
       COUNT(*) FILTER (WHERE ga4_data_start IS NULL) AS null_ga4_start,
       COUNT(*) FILTER (WHERE has_gsc_access IS NULL) AS null_gsc_access
FROM '{CLIENTS}'
""", "Client history coverage")

show(f"""
SELECT access_profile, COUNT(*) AS clients,
       MIN(gsc_data_start) AS min_gsc_start, MAX(gsc_data_start) AS max_gsc_start
FROM '{CLIENTS}' GROUP BY 1 ORDER BY 2 DESC
""", "Clients by access_profile")

# dim_content quick look (no client names, just shape)
show(f"""
SELECT content_type, COUNT(*) AS n,
       COUNT(*) FILTER (WHERE is_published IS TRUE) AS published,
       COUNT(*) FILTER (WHERE is_deleted IS TRUE)   AS deleted,
       COUNT(*) FILTER (WHERE word_count IS NULL)   AS null_word_count
FROM '{CONTENT}' GROUP BY 1 ORDER BY 2 DESC
""", "dim_content by content_type")


=== month=2026-03 summary ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 n_rows   min_date   max_date  contents  clients
9841378 2026-03-01 2026-03-31    331437       55

=== Grain probe (should be empty) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, n]
Index: []

=== Availability flags (month=2026-03) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 gsc_true  gsc_false  gsc_null  ga4_true  ga4_false  ga4_null
  3611061    6230317         0    413966    6408671   3018741

=== Client history coverage ===
 clients earliest_gsc latest_gsc  null_gsc_start  null_ga4_start  null_gsc_access
     104   2025-01-27 2026-06-02              37              53               10

=== Clients by access_profile ===
                      access_profile  clients min_gsc_start max_gsc_start
                         gsc_and_ga4       53    2025-01-27    2026-06-02
       no_search_or_analytics_access       26    2025-11-05    2025-11-05
                            gsc_only       14    2025-06-07    2026-04-07
source_only_missing_client_dimension       10    2025-11-05    2025-11-07
                            ga4_only        1           NaT           NaT

=== dim_content by content_type ===
      content_type      n  published  deleted  null_word_count
   keyword article 459174     361929    96081           174835
    feedly article  57024      46221 

## 3. Methodology

**Label.** At a cutoff date, a page is declining if impressions in the next 30 days are more than 20% below the 30 days before the cutoff. It is computed from daily data, not from the starter file's `trend_direction` or `trend_pct`.

**Windows.** Features use only the 60 days up to the cutoff (two 30-day windows). The label uses the 30 days after it. Dev cutoff: 2026-02-28. June 2026 is a sealed test month, touched once.

**Features.** Impressions, clicks and CTR for both windows, their ratio, average position and its change, share of active days, plus keyword and content properties. Model B adds each feature's percentile rank within its client.

**Baselines.** Two rules with no learning: better average position, and rising impressions momentum.

**Validation.** Client-grouped 5-fold cross-validation, then time-forward tests (train on earlier cutoffs, test on a later one), then the sealed test. **Leakage checks:** no label-window data in any feature, no IDs as features, no trend fields.

*Cell 3a builds the labelled frame for one cutoff. Cell 3b builds features and runs the first client-grouped comparison (development month only).*

In [11]:
import os, duckdb, datetime as dt
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")

BASE = "hf://datasets/FlyRank/internship-warehouse"
CLIENTS = f"{BASE}/dim_clients.parquet"

def show(sql, title=None):
    if title: print(f"\n=== {title} ===")
    print(con.execute(sql).df().to_string(index=False))

def build_frame(cutoff: str, table: str, floor: int = 30, min_days: int = 25):
    C = dt.date.fromisoformat(cutoff)
    d = lambda k: (C + dt.timedelta(days=k)).isoformat()
    # months needed: from C-59 to C+30
    months = sorted({(C + dt.timedelta(days=k)).strftime("%Y-%m") for k in range(-59, 31)})
    files = [f"{BASE}/fact_content_daily_performance/month={m}/data_0.parquet" for m in months]
    print("cutoff", cutoff, "| months:", months)
    con.execute(f"""
    CREATE OR REPLACE TABLE {table} AS
    WITH elig AS (
      SELECT client_hash_id FROM '{CLIENTS}'
      WHERE gsc_data_start IS NOT NULL AND gsc_data_start <= DATE '{d(-59)}'
    ),
    agg AS (
      SELECT f.client_hash_id, f.content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '{d(-29)}' AND DATE '{cutoff}') AS impr_last30,
        SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '{d(-59)}' AND DATE '{d(-30)}') AS impr_prev30,
        SUM(gsc_clicks)      FILTER (WHERE report_date BETWEEN DATE '{d(-29)}' AND DATE '{cutoff}') AS clicks_last30,
        SUM(gsc_clicks)      FILTER (WHERE report_date BETWEEN DATE '{d(-59)}' AND DATE '{d(-30)}') AS clicks_prev30,
        AVG(NULLIF(gsc_avg_position,0)) FILTER (WHERE report_date BETWEEN DATE '{d(-29)}' AND DATE '{cutoff}') AS pos_last30,
        AVG(NULLIF(gsc_avg_position,0)) FILTER (WHERE report_date BETWEEN DATE '{d(-59)}' AND DATE '{d(-30)}') AS pos_prev30,
        COUNT(*) FILTER (WHERE report_date BETWEEN DATE '{d(-29)}' AND DATE '{cutoff}') AS gsc_days_last30,
        COUNT(*) FILTER (WHERE report_date BETWEEN DATE '{d(-59)}' AND DATE '{d(-30)}') AS gsc_days_prev30,
        COUNT(*) FILTER (WHERE report_date BETWEEN DATE '{d(-29)}' AND DATE '{cutoff}' AND gsc_impressions > 0) AS active_days_last30,
        SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '{d(1)}' AND DATE '{d(30)}') AS impr_label,
        COUNT(*) FILTER (WHERE report_date BETWEEN DATE '{d(1)}' AND DATE '{d(30)}') AS gsc_days_label
      FROM read_parquet({files}) f
      JOIN elig USING (client_hash_id)
      WHERE gsc_data_available IS TRUE
        AND report_date BETWEEN DATE '{d(-59)}' AND DATE '{d(30)}'
      GROUP BY 1, 2
    )
    SELECT *, (impr_label < 0.8 * impr_last30)::INT AS is_declining
    FROM agg
    WHERE gsc_days_last30 >= {min_days} AND gsc_days_prev30 >= {min_days}
      AND gsc_days_label >= {min_days} AND impr_last30 >= {floor}
    """)
    show(f"""
    SELECT COUNT(*) AS pages, COUNT(DISTINCT client_hash_id) AS clients,
           ROUND(AVG(is_declining),4) AS base_rate_declining,
           MEDIAN(impr_last30) AS median_impr_last30
    FROM {table}""", f"{table}: size and base rate")
    show(f"""
    SELECT client_hash_id, COUNT(*) AS pages, ROUND(AVG(is_declining),3) AS decl_rate
    FROM {table} GROUP BY 1 ORDER BY 2 DESC LIMIT 10""", f"{table}: top 10 clients by pages (hashed ids)")
    show(f"""
    SELECT MIN(pages) AS min_pages, MEDIAN(pages) AS med_pages, MAX(pages) AS max_pages
    FROM (SELECT COUNT(*) AS pages FROM {table} GROUP BY client_hash_id)""", "pages per client")

build_frame("2026-02-28", "frame_dev")
con.execute("COPY frame_dev TO '/content/frame_dev.parquet' (FORMAT PARQUET)")
print("\nSaved /content/frame_dev.parquet")

cutoff 2026-02-28 | months: ['2025-12', '2026-01', '2026-02', '2026-03']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== frame_dev: size and base rate ===
 pages  clients  base_rate_declining  median_impr_last30
 52774       25               0.2259              1069.0

=== frame_dev: top 10 clients by pages (hashed ids) ===
         client_hash_id  pages  decl_rate
client_73cda7b4e4f265ea  14864      0.247
client_62f4a7e64f5e0096  12278      0.249
client_e547b89c05043229   8078      0.202
client_23a62021009f63c4   6729      0.235
client_fef1a8f436438636   4858      0.111
client_08a6a72ff48e62c0   2644      0.289
client_9958f0a7ae1df715   1456      0.207
client_3197e6291363b4db    643      0.222
client_ff644d8251367cbb    492      0.075
client_65de48885f4ef01b    331      0.227

=== pages per client ===
 min_pages  med_pages  max_pages
         1       67.0      14864

Saved /content/frame_dev.parquet


In [12]:
import os, duckdb, numpy as np, pandas as pd
from google.colab import userdata
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")
BASE = "hf://datasets/FlyRank/internship-warehouse"
CUTOFF = "2026-02-28"

df = con.execute(f"""
SELECT f.client_hash_id, f.content_hash_id, f.is_declining,
  -- window features (all before cutoff)
  LN(1+f.impr_last30) AS log_impr_last30,
  LN(1+f.impr_prev30) AS log_impr_prev30,
  LN((f.impr_last30+1.0)/(f.impr_prev30+1.0)) AS log_impr_ratio,
  LN(1+f.clicks_last30) AS log_clicks_last30,
  f.clicks_last30*1.0/NULLIF(f.impr_last30,0) AS ctr_last30,
  f.clicks_prev30*1.0/NULLIF(f.impr_prev30,0) AS ctr_prev30,
  f.pos_last30, f.pos_last30 - f.pos_prev30 AS pos_delta,
  f.active_days_last30*1.0/f.gsc_days_last30 AS active_day_share,
  (f.impr_prev30 = 0)::INT AS prev_zero,
  -- static content/keyword context (NaN kept, not filled)
  c.content_type, c.main_intent, c.competition_level,
  LN(1+c.search_volume) AS log_search_volume, c.competition, c.cpc,
  c.word_count, c.keyword_token_count, c.url_char_count,
  (c.search_volume IS NOT NULL)::INT AS has_keyword,
  (c.word_count IS NOT NULL)::INT AS has_word_count,
  CASE WHEN c.content_created_date <= DATE '{CUTOFF}'
       THEN DATE_DIFF('day', c.content_created_date, DATE '{CUTOFF}') END AS age_days,
  CASE WHEN c.content_updated_date <= DATE '{CUTOFF}'
       THEN DATE_DIFF('day', c.content_updated_date, DATE '{CUTOFF}') END AS days_since_update
FROM '/content/frame_dev.parquet' f
LEFT JOIN '{BASE}/dim_content.parquet' c
  ON f.client_hash_id = c.client_hash_id AND f.content_hash_id = c.content_hash_id
""").df()
print("rows:", len(df), "| unmatched in dim_content:", df["content_type"].isna().sum())

cats = ["content_type", "main_intent", "competition_level"]
for c in cats:
    df[c] = df[c].astype("category").cat.codes.replace(-1, np.nan)

NUM = ["log_impr_last30","log_impr_prev30","log_impr_ratio","log_clicks_last30","ctr_last30","ctr_prev30",
       "pos_last30","pos_delta","active_day_share","prev_zero","log_search_volume","competition","cpc",
       "word_count","keyword_token_count","url_char_count","has_keyword","has_word_count","age_days","days_since_update"]
FEATS = NUM + cats
X, y, g = df[FEATS].astype(float), df["is_declining"].values, df["client_hash_id"].values
cat_mask = [f in cats for f in FEATS]

def lift_at(y, s, frac=0.10):
    k = max(1, int(len(y)*frac)); top = np.argsort(-s)[:k]
    return y[top].mean(), y[top].mean()/y.mean()

def report(name, y, s):
    p10, l10 = lift_at(y, s)
    return dict(method=name, roc_auc=roc_auc_score(y, s), pr_auc=average_precision_score(y, s),
                prec_top10=p10, lift_top10=l10)

# Same client-grouped folds for baseline and model
oof_model = np.zeros(len(df)); fold_id = np.zeros(len(df), int)
for i, (tr, te) in enumerate(GroupKFold(n_splits=5).split(X, y, g)):
    m = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.06, max_iter=250,
            categorical_features=cat_mask, random_state=42)
    m.fit(X.iloc[tr], y[tr]); oof_model[te] = m.predict_proba(X.iloc[te])[:,1]; fold_id[te] = i

base_score = -df["log_impr_ratio"].values          # bigger = prior momentum already down
rng = np.random.default_rng(42); rand_score = rng.random(len(df))

print(f"\nBase rate declining: {y.mean():.3f}   (pages: {len(y):,}, clients: {len(set(g))})")
res = pd.DataFrame([report("random", y, rand_score),
                    report("baseline: prior 30d momentum", y, base_score),
                    report("model: HistGB (client-grouped 5-fold)", y, oof_model)])
print("\n=== Pooled out-of-fold results (same folds) ===")
print(res.round(3).to_string(index=False))

# Per-client AUC for clients with enough pages and both classes
rows = []
for cl, idx in pd.Series(range(len(df))).groupby(g):
    idx = idx.values
    if len(idx) >= 300 and 0 < y[idx].sum() < len(idx):
        rows.append(dict(client=cl[:15], pages=len(idx), base_rate=y[idx].mean(),
                         auc_baseline=roc_auc_score(y[idx], base_score[idx]),
                         auc_model=roc_auc_score(y[idx], oof_model[idx])))
pc = pd.DataFrame(rows).sort_values("pages", ascending=False)
print("\n=== Per-client AUC (clients with >=300 pages) ===")
print(pc.round(3).to_string(index=False))
print("\nMacro-average AUC  baseline: %.3f | model: %.3f" % (pc.auc_baseline.mean(), pc.auc_model.mean()))

res.to_csv("/content/results_dev.csv", index=False); pc.to_csv("/content/per_client_dev.csv", index=False)
df.assign(oof_model=oof_model).to_parquet("/content/dev_scored.parquet")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows: 52774 | unmatched in dim_content: 0

Base rate declining: 0.226   (pages: 52,774, clients: 25)

=== Pooled out-of-fold results (same folds) ===
                               method  roc_auc  pr_auc  prec_top10  lift_top10
                               random    0.495   0.222       0.212       0.939
         baseline: prior 30d momentum    0.460   0.211       0.208       0.919
model: HistGB (client-grouped 5-fold)    0.646   0.341       0.413       1.827

=== Per-client AUC (clients with >=300 pages) ===
         client  pages  base_rate  auc_baseline  auc_model
client_73cda7b4  14864      0.247         0.415      0.626
client_62f4a7e6  12278      0.249         0.482      0.636
client_e547b89c   8078      0.202         0.457      0.704
client_23a62021   6729      0.235         0.590      0.729
client_fef1a8f4   4858      0.111         0.487      0.625
client_08a6a72f   2644      0.289         0.260      0.347
client_9958f0a7   1456      0.207         0.323      0.703
client_3197

## 4. Results (vs baseline)

**Development months: a null result.** Within one month, client-grouped AUC is about 0.65, but it does not carry forward. Trained on earlier cutoffs and tested on the next, the model (0.564) and the position rule (0.560) are indistinguishable, and adding client-rank features gives 0.580 with macro-client AUC 0.571 against 0.570 for position. Simple rules flipped sign between months.

**Sealed June 2026 (evaluated once): Model B wins.** ROC-AUC 0.640 against 0.469 (position) and 0.516 (momentum), macro-client AUC 0.596, paired client-bootstrap interval for the AUC gain over the position rule [0.120, 0.236]. The June base rate is 61.3%, so precision in the top 10% (78.6%) must be read next to it.

*Cells: 4a single-feature rules, bootstrap interval and importance (dev). 4b second cutoff and time-forward test. 4c third cutoff and client-rank features. 4d the sealed test, which writes `results_sealed.json`.*

In [13]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, average_precision_score

d = pd.read_parquet("/content/dev_scored.parquet")
cats = ["content_type", "main_intent", "competition_level"]
FEATS = ["log_impr_last30","log_impr_prev30","log_impr_ratio","log_clicks_last30","ctr_last30","ctr_prev30",
         "pos_last30","pos_delta","active_day_share","prev_zero","log_search_volume","competition","cpc",
         "word_count","keyword_token_count","url_char_count","has_keyword","has_word_count","age_days",
         "days_since_update"] + cats
y, g, oof = d["is_declining"].values, d["client_hash_id"].values, d["oof_model"].values
X = d[FEATS].astype(float)

# 1. Single-feature baselines (NaN -> median only for this scoring check)
rows = []
for f in FEATS:
    s = X[f].fillna(X[f].median()).values
    a = roc_auc_score(y, s)
    rows.append(dict(feature=f, auc_raw=a, best_direction="higher=decline" if a >= .5 else "lower=decline",
                     auc_best=max(a, 1-a)))
sf = pd.DataFrame(rows).sort_values("auc_best", ascending=False)
print("=== Single-feature baselines (top 10) ===")
print(sf.head(10).round(3).to_string(index=False))
print("\nFlipped momentum AUC: %.3f | Model AUC: %.3f" % (1-roc_auc_score(y, -d.log_impr_ratio), roc_auc_score(y, oof)))

# 2. Cluster bootstrap over clients
rng = np.random.default_rng(42)
clients = np.unique(g); idx_by = {c: np.where(g == c)[0] for c in clients}
boots = []
for _ in range(200):
    pick = rng.choice(clients, len(clients), replace=True)
    ii = np.concatenate([idx_by[c] for c in pick])
    if 0 < y[ii].sum() < len(ii): boots.append(roc_auc_score(y[ii], oof[ii]))
print("\n=== Client-bootstrap 95%% CI for model AUC: [%.3f, %.3f] ===" % tuple(np.percentile(boots, [2.5, 97.5])))

# 3. Permutation importance (AUC drop on held-out client folds)
cat_mask = [f in cats for f in FEATS]
imp = np.zeros(len(FEATS))
for tr, te in GroupKFold(n_splits=5).split(X, y, g):
    m = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.06, max_iter=250,
            categorical_features=cat_mask, random_state=42).fit(X.iloc[tr], y[tr])
    r = permutation_importance(m, X.iloc[te], y[te], scoring="roc_auc", n_repeats=3, random_state=42)
    imp += r.importances_mean / 5
pi = pd.DataFrame({"feature": FEATS, "auc_drop": imp}).sort_values("auc_drop", ascending=False)
print("\n=== Permutation importance (mean AUC drop) ===")
print(pi.round(4).to_string(index=False))

# 4. AUC by impression-volume bucket
d["vol"] = pd.qcut(d["log_impr_last30"], 4, labels=["Q1 low", "Q2", "Q3", "Q4 high"])
print("\n=== Model AUC by volume quartile ===")
for q, sub in d.groupby("vol", observed=True):
    print(f"{q}: pages={len(sub):,} base_rate={sub.is_declining.mean():.3f} "
          f"AUC={roc_auc_score(sub.is_declining, sub.oof_model):.3f}")

sf.to_csv("/content/single_feature_dev.csv", index=False); pi.to_csv("/content/perm_importance_dev.csv", index=False)

=== Single-feature baselines (top 10) ===
            feature  auc_raw best_direction  auc_best
         pos_last30    0.440  lower=decline     0.560
keyword_token_count    0.460  lower=decline     0.540
     log_impr_ratio    0.540 higher=decline     0.540
     has_word_count    0.461  lower=decline     0.539
    log_impr_last30    0.533 higher=decline     0.533
          pos_delta    0.470  lower=decline     0.530
         ctr_last30    0.471  lower=decline     0.529
        competition    0.472  lower=decline     0.528
  log_search_volume    0.478  lower=decline     0.522
     url_char_count    0.484  lower=decline     0.516

Flipped momentum AUC: 0.540 | Model AUC: 0.646

=== Client-bootstrap 95% CI for model AUC: [0.579, 0.694] ===

=== Permutation importance (mean AUC drop) ===
            feature  auc_drop
    log_impr_last30    0.0301
         word_count    0.0299
         pos_last30    0.0285
     log_impr_ratio    0.0278
          pos_delta    0.0199
         ctr_last30    0.

In [14]:
import os, duckdb, datetime as dt, numpy as np, pandas as pd
from google.colab import userdata
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")
BASE = "hf://datasets/FlyRank/internship-warehouse"
CLIENTS = f"{BASE}/dim_clients.parquet"

def build_frame(cutoff, out, floor=30, min_days=25):
    C = dt.date.fromisoformat(cutoff); d = lambda k: (C + dt.timedelta(days=k)).isoformat()
    months = sorted({(C + dt.timedelta(days=k)).strftime("%Y-%m") for k in range(-59, 31)})
    files = [f"{BASE}/fact_content_daily_performance/month={m}/data_0.parquet" for m in months]
    print("cutoff", cutoff, "| months:", months)
    con.execute(f"""
    COPY (
    WITH elig AS (SELECT client_hash_id FROM '{CLIENTS}'
                  WHERE gsc_data_start IS NOT NULL AND gsc_data_start <= DATE '{d(-59)}'),
    agg AS (
      SELECT f.client_hash_id, f.content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '{d(-29)}' AND DATE '{cutoff}') AS impr_last30,
        SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '{d(-59)}' AND DATE '{d(-30)}') AS impr_prev30,
        SUM(gsc_clicks) FILTER (WHERE report_date BETWEEN DATE '{d(-29)}' AND DATE '{cutoff}') AS clicks_last30,
        SUM(gsc_clicks) FILTER (WHERE report_date BETWEEN DATE '{d(-59)}' AND DATE '{d(-30)}') AS clicks_prev30,
        AVG(NULLIF(gsc_avg_position,0)) FILTER (WHERE report_date BETWEEN DATE '{d(-29)}' AND DATE '{cutoff}') AS pos_last30,
        AVG(NULLIF(gsc_avg_position,0)) FILTER (WHERE report_date BETWEEN DATE '{d(-59)}' AND DATE '{d(-30)}') AS pos_prev30,
        COUNT(*) FILTER (WHERE report_date BETWEEN DATE '{d(-29)}' AND DATE '{cutoff}') AS gsc_days_last30,
        COUNT(*) FILTER (WHERE report_date BETWEEN DATE '{d(-59)}' AND DATE '{d(-30)}') AS gsc_days_prev30,
        COUNT(*) FILTER (WHERE report_date BETWEEN DATE '{d(-29)}' AND DATE '{cutoff}' AND gsc_impressions > 0) AS active_days_last30,
        SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '{d(1)}' AND DATE '{d(30)}') AS impr_label,
        COUNT(*) FILTER (WHERE report_date BETWEEN DATE '{d(1)}' AND DATE '{d(30)}') AS gsc_days_label
      FROM read_parquet({files}) f JOIN elig USING (client_hash_id)
      WHERE gsc_data_available IS TRUE AND report_date BETWEEN DATE '{d(-59)}' AND DATE '{d(30)}'
      GROUP BY 1, 2)
    SELECT *, (impr_label < 0.8 * impr_last30)::INT AS is_declining FROM agg
    WHERE gsc_days_last30 >= {min_days} AND gsc_days_prev30 >= {min_days}
      AND gsc_days_label >= {min_days} AND impr_last30 >= {floor}
    ) TO '{out}' (FORMAT PARQUET)""")

CAT_LEVELS = {"content_type": ["keyword article","feedly article","comparison article"],
              "main_intent": ["informational","transactional","commercial","navigational"],
              "competition_level": ["LOW","MEDIUM","HIGH"]}
NUM = ["log_impr_last30","log_impr_prev30","log_impr_ratio","log_clicks_last30","ctr_last30","ctr_prev30",
       "pos_last30","pos_delta","active_day_share","prev_zero","log_search_volume","competition","cpc",
       "word_count","keyword_token_count","url_char_count","has_keyword","has_word_count","age_days","days_since_update"]
CATS = list(CAT_LEVELS); FEATS = NUM + CATS

def make_features(path, cutoff):
    df = con.execute(f"""
    SELECT f.client_hash_id, f.content_hash_id, f.is_declining,
      LN(1+f.impr_last30) AS log_impr_last30, LN(1+f.impr_prev30) AS log_impr_prev30,
      LN((f.impr_last30+1.0)/(f.impr_prev30+1.0)) AS log_impr_ratio, LN(1+f.clicks_last30) AS log_clicks_last30,
      f.clicks_last30*1.0/NULLIF(f.impr_last30,0) AS ctr_last30, f.clicks_prev30*1.0/NULLIF(f.impr_prev30,0) AS ctr_prev30,
      f.pos_last30, f.pos_last30 - f.pos_prev30 AS pos_delta,
      f.active_days_last30*1.0/f.gsc_days_last30 AS active_day_share, (f.impr_prev30 = 0)::INT AS prev_zero,
      c.content_type, c.main_intent, c.competition_level, LN(1+c.search_volume) AS log_search_volume,
      c.competition, c.cpc, c.word_count, c.keyword_token_count, c.url_char_count,
      (c.search_volume IS NOT NULL)::INT AS has_keyword, (c.word_count IS NOT NULL)::INT AS has_word_count,
      CASE WHEN c.content_created_date <= DATE '{cutoff}' THEN DATE_DIFF('day', c.content_created_date, DATE '{cutoff}') END AS age_days,
      CASE WHEN c.content_updated_date <= DATE '{cutoff}' THEN DATE_DIFF('day', c.content_updated_date, DATE '{cutoff}') END AS days_since_update
    FROM '{path}' f LEFT JOIN '{BASE}/dim_content.parquet' c
      ON f.client_hash_id=c.client_hash_id AND f.content_hash_id=c.content_hash_id""").df()
    for c, lv in CAT_LEVELS.items():
        df[c] = pd.Categorical(df[c], categories=lv).codes.astype(float)
        df.loc[df[c] < 0, c] = np.nan
    return df

def fit(X, y, feats):
    return HistGradientBoostingClassifier(max_depth=4, learning_rate=0.06, max_iter=250,
        categorical_features=[f in CATS for f in feats], random_state=42).fit(X[feats].astype(float), y)

def lift10(y, s):
    k = int(len(y)*0.10); top = np.argsort(-s)[:k]; return y[top].mean(), y[top].mean()/y.mean()

def summarize(name, y, s):
    p, l = lift10(y, s)
    return dict(method=name, roc_auc=roc_auc_score(y, s), pr_auc=average_precision_score(y, s), prec_top10=p, lift_top10=l)

# Build second dev frame (label = Feb 2026) and features for both cutoffs
build_frame("2026-01-31", "/content/frame_dev2.parquet")
d2 = make_features("/content/frame_dev2.parquet", "2026-01-31")
d1 = make_features("/content/frame_dev.parquet", "2026-02-28")
print("\nframe2 (cutoff Jan-31): pages", len(d2), "clients", d2.client_hash_id.nunique(), "base_rate %.3f" % d2.is_declining.mean())
print("frame1 (cutoff Feb-28): pages", len(d1), "clients", d1.client_hash_id.nunique(), "base_rate %.3f" % d1.is_declining.mean())

# (a) Stability: client-grouped CV on frame2
y2, g2 = d2.is_declining.values, d2.client_hash_id.values
oof2 = np.zeros(len(d2))
for tr, te in GroupKFold(5).split(d2, y2, g2):
    oof2[te] = fit(d2.iloc[tr], y2[tr], FEATS).predict_proba(d2.iloc[te][FEATS].astype(float))[:,1]
print("\n=== (a) Client-grouped CV on frame2 (cutoff Jan-31) ===")
print(pd.DataFrame([summarize("model", y2, oof2),
                    summarize("baseline: rising momentum", y2, d2.log_impr_ratio.values),
                    summarize("baseline: better position", y2, -d2.pos_last30.fillna(d2.pos_last30.median()).values)]).round(3).to_string(index=False))

# (b) Time-forward: train on frame2 (labels end Feb-28) -> test on frame1 (labels Mar)
y1, g1 = d1.is_declining.values, d1.client_hash_id.values
train_clients, test_clients = set(g2), set(g1)
print("\nClient overlap train/test: %d of %d test clients appear in train" % (len(train_clients & test_clients), len(test_clients)))
m_full = fit(d2, y2, FEATS); s_full = m_full.predict_proba(d1[FEATS].astype(float))[:,1]
F_NOWC = [f for f in FEATS if f not in ("word_count","has_word_count")]
s_nowc = fit(d2, y2, F_NOWC).predict_proba(d1[F_NOWC].astype(float))[:,1]
print("\n=== (b) Time-forward: train Jan-31 cutoff -> test Feb-28 cutoff (base rate %.3f) ===" % y1.mean())
res = pd.DataFrame([summarize("random", y1, np.random.default_rng(42).random(len(y1))),
                    summarize("baseline: rising momentum", y1, d1.log_impr_ratio.values),
                    summarize("baseline: better position", y1, -d1.pos_last30.fillna(d1.pos_last30.median()).values),
                    summarize("model: all features", y1, s_full),
                    summarize("model: without word_count", y1, s_nowc)])
print(res.round(3).to_string(index=False))

# Per-client AUC in forward test (clients with >=300 pages)
rows = []
for cl in pd.unique(g1):
    idx = np.where(g1 == cl)[0]
    if len(idx) >= 300 and 0 < y1[idx].sum() < len(idx):
        rows.append(dict(client=cl[:15], pages=len(idx), base_rate=y1[idx].mean(),
                         auc_model=roc_auc_score(y1[idx], s_full[idx]), auc_no_wc=roc_auc_score(y1[idx], s_nowc[idx])))
pc = pd.DataFrame(rows).sort_values("pages", ascending=False)
print("\n=== Per-client forward AUC ===")
print(pc.round(3).to_string(index=False))
print("Macro AUC: model %.3f | without word_count %.3f" % (pc.auc_model.mean(), pc.auc_no_wc.mean()))
res.to_csv("/content/results_forward.csv", index=False)

cutoff 2026-01-31 | months: ['2025-12', '2026-01', '2026-02', '2026-03']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


frame2 (cutoff Jan-31): pages 45897 clients 23 base_rate 0.146
frame1 (cutoff Feb-28): pages 52774 clients 25 base_rate 0.226

=== (a) Client-grouped CV on frame2 (cutoff Jan-31) ===
                   method  roc_auc  pr_auc  prec_top10  lift_top10
                    model    0.647   0.269       0.380       2.607
baseline: rising momentum    0.440   0.166       0.220       1.508
baseline: better position    0.490   0.147       0.154       1.057

Client overlap train/test: 23 of 25 test clients appear in train

=== (b) Time-forward: train Jan-31 cutoff -> test Feb-28 cutoff (base rate 0.226) ===
                   method  roc_auc  pr_auc  prec_top10  lift_top10
                   random    0.491   0.222       0.218       0.963
baseline: rising momentum    0.540   0.261       0.300       1.329
baseline: better position    0.560   0.249       0.251       1.112
      model: all features    0.564   0.272       0.302       1.338
model: without word_count    0.542   0.248       0.266      

In [15]:
build_frame("2025-12-31", "/content/frame_dev3.parquet")
d3 = make_features("/content/frame_dev3.parquet", "2025-12-31")
print("frame3 (cutoff Dec-31): pages", len(d3), "clients", d3.client_hash_id.nunique(), "base_rate %.3f" % d3.is_declining.mean())

REL_BASE = ["log_impr_last30","log_impr_ratio","pos_last30","ctr_last30","pos_delta","log_clicks_last30"]
def add_rel(df):
    df = df.copy()
    for f in REL_BASE:
        df[f+"_crank"] = df.groupby("client_hash_id")[f].rank(pct=True)   # within-client percentile at cutoff
    return df
d1r, d2r, d3r = add_rel(d1), add_rel(d2), add_rel(d3)
REL = [f+"_crank" for f in REL_BASE]

SETS = {
  "A: original features": FEATS,
  "B: original + client-rank": FEATS + REL,
  "C: client-rank only (no word_count)": REL + ["prev_zero","active_day_share","keyword_token_count","url_char_count",
                                               "age_days","days_since_update"] + CATS,
}
train = pd.concat([d3r, d2r], ignore_index=True)          # labels end Feb-28 at the latest
ytr, y1, g1 = train.is_declining.values, d1r.is_declining.values, d1r.client_hash_id.values
print("train pages %d | test pages %d (base rate %.3f)" % (len(train), len(d1r), y1.mean()))

def macro_within_client(y, s, g, min_pages=300):
    out = []
    for cl in pd.unique(g):
        idx = np.where(g == cl)[0]
        if len(idx) >= min_pages and 0 < y[idx].sum() < len(idx):
            out.append(roc_auc_score(y[idx], s[idx]))
    return np.mean(out), len(out), np.mean(np.array(out) > 0.5)

pos_s = -d1r.pos_last30.fillna(d1r.pos_last30.median()).values
scores = {"baseline: better position": pos_s,
          "baseline: rising momentum": d1r.log_impr_ratio.values}
for name, feats in SETS.items():
    m = fit(train, ytr, feats)
    scores["model " + name] = m.predict_proba(d1r[feats].astype(float))[:, 1]

rows = []
for name, s in scores.items():
    r = summarize(name, y1, s)
    ma, n, frac = macro_within_client(y1, s, g1)
    r.update(macro_client_auc=ma, share_clients_above_0_5=frac)
    rows.append(r)
res = pd.DataFrame(rows)
print("\n=== Train Dec-31 + Jan-31 cutoffs -> test Feb-28 cutoff ===")
print(res.round(3).to_string(index=False))
res.to_csv("/content/results_forward2.csv", index=False)

cutoff 2025-12-31 | months: ['2025-11', '2025-12', '2026-01']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

frame3 (cutoff Dec-31): pages 33008 clients 20 base_rate 0.178
train pages 78905 | test pages 52774 (base rate 0.226)

=== Train Dec-31 + Jan-31 cutoffs -> test Feb-28 cutoff ===
                                   method  roc_auc  pr_auc  prec_top10  lift_top10  macro_client_auc  share_clients_above_0_5
                baseline: better position    0.560   0.249       0.251       1.112             0.570                      0.7
                baseline: rising momentum    0.540   0.261       0.300       1.329             0.575                      0.9
               model A: original features    0.568   0.260       0.264       1.170             0.556                      0.8
          model B: original + client-rank    0.580   0.267       0.285       1.263             0.571                      0.8
model C: client-rank only (no word_count)    0.534   0.249       0.252       1.115             0.554                      0.8


In [16]:
import json, datetime as dt

# --- build the extra training frames + the sealed frame (one scan each) ---
for cut, path in [("2026-03-31", "/content/frame_dev4.parquet"),
                  ("2026-04-30", "/content/frame_dev5.parquet"),
                  ("2026-05-31", "/content/frame_SEALED_june.parquet")]:
    build_frame(cut, path)

d4r = add_rel(make_features("/content/frame_dev4.parquet", "2026-03-31"))
d5r = add_rel(make_features("/content/frame_dev5.parquet", "2026-04-30"))
dS  = add_rel(make_features("/content/frame_SEALED_june.parquet", "2026-05-31"))

FEATS_B = FEATS + REL
train = pd.concat([d3r, d2r, d1r, d4r, d5r], ignore_index=True)   # cutoffs Dec-31 ... Apr-30
ytr = train.is_declining.values
yS, gS = dS.is_declining.values, dS.client_hash_id.values
print("train pages %d | SEALED pages %d, clients %d, base rate %.3f" %
      (len(train), len(dS), dS.client_hash_id.nunique(), yS.mean()))

# --- frozen candidates ---
pos_S  = -dS.pos_last30.fillna(dS.pos_last30.median()).values
mom_S  = dS.log_impr_ratio.values
modelB = fit(train, ytr, FEATS_B)
mB_S   = modelB.predict_proba(dS[FEATS_B].astype(float))[:, 1]
rand_S = np.random.default_rng(42).random(len(dS))

rows = []
for name, s in [("random", rand_S), ("baseline: better position", pos_S),
                ("baseline: rising momentum", mom_S), ("model B", mB_S)]:
    r = summarize(name, yS, s)
    ma, n, frac = macro_within_client(yS, s, gS)
    r.update(macro_client_auc=ma, n_clients_scored=n, share_clients_above_0_5=frac)
    rows.append(r)
res = pd.DataFrame(rows)
print("\n=== SEALED June-2026 test (base rate %.3f) ===" % yS.mean())
print(res.round(3).to_string(index=False))

# --- client-bootstrap CI: Model B minus position rule (paired) ---
rng = np.random.default_rng(42)
cl_all = np.unique(gS); idx_by = {c: np.where(gS == c)[0] for c in cl_all}
diffs = []
for _ in range(300):
    ii = np.concatenate([idx_by[c] for c in rng.choice(cl_all, len(cl_all), replace=True)])
    if 0 < yS[ii].sum() < len(ii):
        diffs.append(roc_auc_score(yS[ii], mB_S[ii]) - roc_auc_score(yS[ii], pos_S[ii]))
ci = np.percentile(diffs, [2.5, 97.5])
print("\nModel B minus position rule, AUC diff 95%% client-bootstrap CI: [%.3f, %.3f]" % tuple(ci))

# --- save the checkable evidence ---
out = {"cutoff": "2026-05-31", "label_window": "2026-06-01..2026-06-30",
       "train_cutoffs": ["2025-12-31","2026-01-31","2026-02-28","2026-03-31","2026-04-30"],
       "pages": int(len(dS)), "clients": int(dS.client_hash_id.nunique()),
       "base_rate": float(yS.mean()), "results": res.round(4).to_dict(orient="records"),
       "auc_diff_modelB_minus_position_CI95": [float(ci[0]), float(ci[1])], "seed": 42}
json.dump(out, open("/content/results_sealed.json", "w"), indent=2)
res.to_csv("/content/results_sealed.csv", index=False)
print("saved results_sealed.json / .csv")

cutoff 2026-03-31 | months: ['2026-01', '2026-02', '2026-03', '2026-04']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

cutoff 2026-04-30 | months: ['2026-03', '2026-04', '2026-05']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

cutoff 2026-05-31 | months: ['2026-04', '2026-05', '2026-06']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

train pages 263952 | SEALED pages 71792, clients 35, base rate 0.613

=== SEALED June-2026 test (base rate 0.613) ===
                   method  roc_auc  pr_auc  prec_top10  lift_top10  macro_client_auc  n_clients_scored  share_clients_above_0_5
                   random    0.500   0.613       0.615       1.004             0.501                19                    0.474
baseline: better position    0.469   0.572       0.471       0.768             0.467                19                    0.316
baseline: rising momentum    0.516   0.615       0.596       0.973             0.526                19                    0.737
                  model B    0.640   0.721       0.786       1.284             0.596                19                    0.895

Model B minus position rule, AUC diff 95% client-bootstrap CI: [0.120, 0.236]
saved results_sealed.json / .csv


## 5. Limitations

- **One sealed month.** The win is observed on a single label window and the development months were near-null, so it is not shown to be stable.
- **Regime shift.** The share of declining pages rose from 14.6% to 61.3% across label months. Total impressions fell about 25% from May to June.
- **Between-client differences** contribute to the pooled AUC. The macro-client figure (0.596) is the conservative number.
- **Ranking, not probability.** Mean predicted risk was 0.541 against an actual 0.613.
- **Unstable drivers.** Importance and simple-rule direction changed between months.
- **Not causal.** Reason codes describe a page's features and do not explain why it fell. Nothing here says anything about how Google's algorithm works.
- **The live list is a deployment refit** on all labelled cutoffs including June, so it has no held-out score. Tier cut-offs are judgment calls.

*The cell below is post-hoc diagnostics that back these points. It does not change the sealed result.*

In [21]:
from sklearn.inspection import permutation_importance

# 1. Where does the 61% come from? Per-client base rate + aggregate traffic change
dS["client"] = dS.client_hash_id.str[:15]
pc = (dS.groupby("client").agg(pages=("is_declining","size"), base_rate=("is_declining","mean"))
        .sort_values("pages", ascending=False))
print("=== Sealed: per-client base rate (top 12 by pages) ===")
print(pc.head(12).round(3).to_string())
print("Share of pages in clients with base_rate > 0.8: %.3f" % (pc.loc[pc.base_rate>0.8,"pages"].sum()/pc.pages.sum()))

raw = pd.read_parquet("/content/frame_SEALED_june.parquet")
print("\nAggregate impressions  May(last30): %d | June(label): %d | ratio %.3f" %
      (raw.impr_last30.sum(), raw.impr_label.sum(), raw.impr_label.sum()/raw.impr_last30.sum()))
print("Median page June/May ratio: %.3f" % (raw.impr_label/raw.impr_last30).median())

# 2. Does the win survive without the two biggest clients?
big = pc.index[:2]
mask = ~dS.client.isin(big).values
print("\nAUC excluding top-2 clients: model B %.3f | position %.3f | momentum %.3f (pages %d)" % (
    roc_auc_score(yS[mask], mB_S[mask]), roc_auc_score(yS[mask], pos_S[mask]),
    roc_auc_score(yS[mask], mom_S[mask]), mask.sum()))

# 3. Calibration reminder
print("Mean predicted P(decline): %.3f vs actual base rate %.3f" % (mB_S.mean(), yS.mean()))

# 4. Post-hoc permutation importance for Model B (interpretation only, not model selection)
Xs = dS[FEATS_B].astype(float)
r = permutation_importance(modelB, Xs, yS, scoring="roc_auc", n_repeats=3, random_state=42)
pi = pd.DataFrame({"feature": FEATS_B, "auc_drop": r.importances_mean}).sort_values("auc_drop", ascending=False)
print("\n=== Model B permutation importance on sealed month (post-hoc) ===")
print(pi.head(12).round(4).to_string(index=False))

pc.to_csv("/content/sealed_per_client_base_rate.csv"); pi.to_csv("/content/perm_importance_modelB.csv", index=False)

=== Sealed: per-client base rate (top 12 by pages) ===
                 pages  base_rate
client                           
client_73cda7b4  18599      0.714
client_62f4a7e6  11513      0.557
client_23a62021  11200      0.727
client_e547b89c   7743      0.307
client_fef1a8f4   3650      0.738
client_08a6a72f   3589      0.687
client_20259bd6   3167      0.794
client_e5c2aa26   2791      0.734
client_a80fca3f   1816      0.300
client_3f0ce4d4   1352      0.834
client_1a730cb2   1324      0.245
client_0fa64a18    823      0.275
Share of pages in clients with base_rate > 0.8: 0.021

Aggregate impressions  May(last30): 215612582 | June(label): 162066806 | ratio 0.752
Median page June/May ratio: 0.674

AUC excluding top-2 clients: model B 0.631 | position 0.436 | momentum 0.490 (pages 41680)
Mean predicted P(decline): 0.541 vs actual base rate 0.613

=== Model B permutation importance on sealed month (post-hoc) ===
                feature  auc_drop
        log_impr_last30    0.0632
        l

## 6. Ranked recommendations

Every eligible page as of 2026-06-30 is scored, gated on risk first, then ordered by traffic within a tier. **Tier A** (review first): risk in the top 20% and traffic above the client's median, with at most 25% from one client before demotion. **Tier B** (review): risk in the top 30%. **Tier C**: monitor.

Suggested review comes from the first matching reason code: position slip, low CTR for its client, striking distance (position 8 to 20), otherwise a general high-risk review. Reason codes are descriptions, not causes.

The full page-level list contains pseudonymous ids and stays private. Only tier and reason-code summaries are published.

*Cell 6a scores the live pages and assigns reason codes. Cell 6b re-ranks (risk gate first) and saves the summaries.*

In [22]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, shutil

# ---------- 1. Score frame at cutoff 2026-06-30 (features only, no label window) ----------
CUT = "2026-06-30"
C = dt.date.fromisoformat(CUT); dd = lambda k: (C + dt.timedelta(days=k)).isoformat()
months = sorted({(C + dt.timedelta(days=k)).strftime("%Y-%m") for k in range(-59, 1)})
files = [f"{BASE}/fact_content_daily_performance/month={m}/data_0.parquet" for m in months]
print("scoring cutoff", CUT, "| months:", months)
con.execute(f"""
COPY (
WITH elig AS (SELECT client_hash_id FROM '{CLIENTS}'
              WHERE gsc_data_start IS NOT NULL AND gsc_data_start <= DATE '{dd(-59)}'),
agg AS (
  SELECT f.client_hash_id, f.content_hash_id,
    SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '{dd(-29)}' AND DATE '{CUT}') AS impr_last30,
    SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '{dd(-59)}' AND DATE '{dd(-30)}') AS impr_prev30,
    SUM(gsc_clicks) FILTER (WHERE report_date BETWEEN DATE '{dd(-29)}' AND DATE '{CUT}') AS clicks_last30,
    SUM(gsc_clicks) FILTER (WHERE report_date BETWEEN DATE '{dd(-59)}' AND DATE '{dd(-30)}') AS clicks_prev30,
    AVG(NULLIF(gsc_avg_position,0)) FILTER (WHERE report_date BETWEEN DATE '{dd(-29)}' AND DATE '{CUT}') AS pos_last30,
    AVG(NULLIF(gsc_avg_position,0)) FILTER (WHERE report_date BETWEEN DATE '{dd(-59)}' AND DATE '{dd(-30)}') AS pos_prev30,
    COUNT(*) FILTER (WHERE report_date BETWEEN DATE '{dd(-29)}' AND DATE '{CUT}') AS gsc_days_last30,
    COUNT(*) FILTER (WHERE report_date BETWEEN DATE '{dd(-59)}' AND DATE '{dd(-30)}') AS gsc_days_prev30,
    COUNT(*) FILTER (WHERE report_date BETWEEN DATE '{dd(-29)}' AND DATE '{CUT}' AND gsc_impressions > 0) AS active_days_last30
  FROM read_parquet({files}) f JOIN elig USING (client_hash_id)
  WHERE gsc_data_available IS TRUE AND report_date BETWEEN DATE '{dd(-59)}' AND DATE '{CUT}'
  GROUP BY 1, 2)
SELECT *, 0 AS is_declining FROM agg
WHERE gsc_days_last30 >= 25 AND gsc_days_prev30 >= 25 AND impr_last30 >= 30
) TO '/content/frame_SCORE_jun30.parquet' (FORMAT PARQUET)""")

dL = add_rel(make_features("/content/frame_SCORE_jun30.parquet", CUT))
print("live pages:", len(dL), "| clients:", dL.client_hash_id.nunique())

# ---------- 2. Deployment refit on all labeled cutoffs (labels end 2026-06-30) ----------
train_all = pd.concat([d3r, d2r, d1r, d4r, d5r, dS], ignore_index=True)
m_final = fit(train_all, train_all.is_declining.values, FEATS_B)
dL["risk_score"] = m_final.predict_proba(dL[FEATS_B].astype(float))[:, 1]
dL["risk_pct"] = dL.risk_score.rank(pct=True)
dL["impr_last30"] = np.expm1(dL.log_impr_last30).round().astype(int)
dL["impr_at_risk"] = dL.risk_pct * dL.impr_last30          # rank x traffic at stake (ordering only)

# ---------- 3. Reason codes (descriptive feature flags, not causes) ----------
flags = {
  "HIGH_TRAFFIC":     dL.log_impr_last30_crank >= 0.90,
  "RECENT_SURGE":     dL.log_impr_ratio_crank  >= 0.80,
  "MOMENTUM_DOWN":    dL.log_impr_ratio_crank  <= 0.20,
  "POSITION_SLIP":    dL.pos_delta >= 2,
  "STRIKING_DISTANCE": dL.pos_last30.between(8, 20, inclusive="right"),
  "LOW_CTR_FOR_CLIENT": (dL.ctr_last30_crank <= 0.25) & (dL.log_impr_last30_crank >= 0.5),
}
for k, v in flags.items(): dL[k] = v.fillna(False)
dL["reason_codes"] = dL[list(flags)].apply(lambda r: "|".join(k for k in flags if r[k]) or "NONE", axis=1)

def action(r):
    if r.POSITION_SLIP:        return "Review content freshness / SERP change"
    if r.LOW_CTR_FOR_CLIENT:   return "Review title & meta description"
    if r.STRIKING_DISTANCE:    return "Improve content depth (near page 1)"
    if r.MOMENTUM_DOWN:        return "Monitor next 2 weeks"
    return "Monitor"
dL["suggested_review"] = dL.apply(action, axis=1)

q = dL.impr_at_risk.rank(pct=True)
dL["priority_tier"] = pd.cut(q, [0, .80, .95, 1.0], labels=["C: monitor", "B: review", "A: review first"])
dL = dL.sort_values("impr_at_risk", ascending=False).reset_index(drop=True)
dL["rank"] = np.arange(1, len(dL) + 1)

print("\n=== Priority tiers ===");  print(dL.priority_tier.value_counts().to_string())
print("\n=== Reason-code frequency (all pages) ===")
print(pd.Series({k: int(dL[k].sum()) for k in flags}).to_string())
top500 = dL.head(500).client_hash_id.value_counts(normalize=True)
print("\nTop-500 pages: share from largest client %.2f | clients represented %d" % (top500.iloc[0], len(top500)))
show_cols = ["rank","risk_pct","impr_last30","reason_codes","suggested_review"]
ex = dL.head(10)[show_cols].copy(); ex["risk_pct"] = ex.risk_pct.round(3)
print("\n=== Top 10 (ids omitted) ==="); print(ex.to_string(index=False))

# ---------- 4. Save outputs (full list stays LOCAL; only the public-safe summary is for the repo) ----------
dL[["client_hash_id","content_hash_id","rank","priority_tier","risk_pct","impr_last30","reason_codes","suggested_review"]] \
  .to_csv("/content/ranked_pages_jun30_PRIVATE.csv", index=False)
summary = dL.groupby("priority_tier", observed=True).agg(pages=("rank","size"), median_impr=("impr_last30","median")).reset_index()
summary.to_csv("/content/playbook_summary.csv", index=False)

scoring cutoff 2026-06-30 | months: ['2026-05', '2026-06']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

live pages: 84190 | clients: 41

=== Priority tiers ===
priority_tier
C: monitor         67352
B: review          12628
A: review first     4210

=== Reason-code frequency (all pages) ===
HIGH_TRAFFIC           8446
RECENT_SURGE          16865
MOMENTUM_DOWN         16825
POSITION_SLIP         25312
STRIKING_DISTANCE     31895
LOW_CTR_FOR_CLIENT     5511

Top-500 pages: share from largest client 0.37 | clients represented 20

=== Top 10 (ids omitted) ===
 rank  risk_pct  impr_last30                                                                 reason_codes                       suggested_review
    1     0.350       585712                                                    HIGH_TRAFFIC|RECENT_SURGE                                Monitor
    2     0.965       179662 HIGH_TRAFFIC|RECENT_SURGE|POSITION_SLIP|STRIKING_DISTANCE|LOW_CTR_FOR_CLIENT Review content freshness / SERP change
    3     0.993       169233                                                                 HIGH_TRAFFIC  

In [23]:
# ---------- Re-rank: gate on risk, then order by traffic ----------
dL["tier"] = "C: monitor"
dL.loc[dL.risk_pct >= 0.70, "tier"] = "B: review"
dL.loc[(dL.risk_pct >= 0.80) & (dL.log_impr_last30_crank >= 0.5), "tier"] = "A: review first"

def action2(r):
    if r.POSITION_SLIP:         return "Review content freshness / SERP change"
    if r.LOW_CTR_FOR_CLIENT:    return "Review title & meta description"
    if r.STRIKING_DISTANCE:     return "Improve content depth (near page 1)"
    if r.tier.startswith("A"):  return "Review page: high risk rank, high traffic"
    if r.MOMENTUM_DOWN:         return "Monitor next 2 weeks"
    return "Monitor"
dL["suggested_review"] = dL.apply(action2, axis=1)

# order: tier, then traffic x risk inside the tier
dL["tier_order"] = dL.tier.map({"A: review first": 0, "B: review": 1, "C: monitor": 2})
dL = dL.sort_values(["tier_order", "impr_at_risk"], ascending=[True, False]).reset_index(drop=True)

# cap: at most 25% of Tier A from any one client (demote the excess to Tier B)
A = dL[dL.tier.str.startswith("A")]
cap = int(0.25 * len(A))
excess_idx = []
for cl, sub in A.groupby("client_hash_id"):
    if len(sub) > cap:
        excess_idx += list(sub.index[cap:])
dL.loc[excess_idx, "tier"] = "B: review"
dL["tier_order"] = dL.tier.map({"A: review first": 0, "B: review": 1, "C: monitor": 2})
dL = dL.sort_values(["tier_order", "impr_at_risk"], ascending=[True, False]).reset_index(drop=True)
dL["rank"] = np.arange(1, len(dL) + 1)
dL["priority_tier"] = dL.tier

print("=== Tiers ==="); print(dL.priority_tier.value_counts().to_string())
A = dL[dL.priority_tier.str.startswith("A")]
print("\nTier A: pages %d | clients %d | largest client share %.2f | median risk_pct %.3f | median impr_last30 %d" % (
    len(A), A.client_hash_id.nunique(), A.client_hash_id.value_counts(normalize=True).iloc[0],
    A.risk_pct.median(), A.impr_last30.median()))
print("\nTier A reason-code frequency:")
print(pd.Series({k: round(A[k].mean(), 3) for k in flags}).to_string())
print("\nTier A suggested review mix:"); print(A.suggested_review.value_counts().to_string())
ex = dL.head(10)[["rank","risk_pct","impr_last30","reason_codes","suggested_review"]].copy()
ex["risk_pct"] = ex.risk_pct.round(3)
print("\n=== New top 10 (ids omitted) ==="); print(ex.to_string(index=False))

# save (private full list stays local; only the summary goes public)
dL[["client_hash_id","content_hash_id","rank","priority_tier","risk_pct","impr_last30","reason_codes","suggested_review"]] \
  .to_csv("/content/ranked_pages_jun30_PRIVATE.csv", index=False)
summary = (dL.groupby("priority_tier", observed=True)
             .agg(pages=("rank","size"), median_impr_last30=("impr_last30","median"), median_risk_pct=("risk_pct","median"))
             .reset_index())
summary.to_csv("/content/playbook_summary.csv", index=False)
rc = pd.DataFrame({"reason_code": list(flags), "share_of_tier_A": [A[k].mean() for k in flags],
                   "share_of_all_pages": [dL[k].mean() for k in flags]})
rc.to_csv("/content/reason_code_summary.csv", index=False)
print("\nsaved playbook_summary.csv, reason_code_summary.csv")

=== Tiers ===
priority_tier
C: monitor         58932
B: review          13775
A: review first    11483

Tier A: pages 11483 | clients 33 | largest client share 0.27 | median risk_pct 0.910 | median impr_last30 1587

Tier A reason-code frequency:
HIGH_TRAFFIC          0.183
RECENT_SURGE          0.157
MOMENTUM_DOWN         0.164
POSITION_SLIP         0.359
STRIKING_DISTANCE     0.469
LOW_CTR_FOR_CLIENT    0.279

Tier A suggested review mix:
suggested_review
Review content freshness / SERP change       4117
Review page: high risk rank, high traffic    2814
Improve content depth (near page 1)          2711
Review title & meta description              1841

=== New top 10 (ids omitted) ===
 rank  risk_pct  impr_last30                                                                 reason_codes                          suggested_review
    1     0.965       179662 HIGH_TRAFFIC|RECENT_SURGE|POSITION_SLIP|STRIKING_DISTANCE|LOW_CTR_FOR_CLIENT    Review content freshness / SERP change
    2    

## 7. Artifacts the paper embeds

The three charts on the paper (sealed AUC, base rate by month, feature importance) and a zip of the public-safe CSV/JSON files. The private ranked list is deliberately excluded.

In [24]:
# ---------- 5. Charts for the paper ----------
rs = pd.read_csv("/content/results_sealed.csv"); pim = pd.read_csv("/content/perm_importance_modelB.csv").head(8)
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.barh(rs.method[::-1], rs.roc_auc[::-1], color=["#2a6f97" if "model" in m else "#9aa5b1" for m in rs.method[::-1]])
ax.axvline(0.5, ls="--", c="k", lw=1); ax.set_xlim(0.4, 0.7); ax.set_xlabel("ROC-AUC on sealed June 2026 (0.5 = chance)")
plt.tight_layout(); plt.savefig("/content/fig1_sealed_auc.png", dpi=160); plt.close()

br = {"Jan label": d3r.is_declining.mean(), "Feb label": d2r.is_declining.mean(), "Mar label": d1r.is_declining.mean(),
      "Apr label": d4r.is_declining.mean(), "May label": d5r.is_declining.mean(), "Jun label (sealed)": dS.is_declining.mean()}
print("\nBase rate by label month:", {k: round(v, 3) for k, v in br.items()})
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.bar(br.keys(), br.values(), color="#2a6f97"); ax.set_ylabel("Share of pages declining >20%")
plt.xticks(rotation=25, ha="right"); plt.tight_layout(); plt.savefig("/content/fig2_base_rate_by_month.png", dpi=160); plt.close()

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.barh(pim.feature[::-1], pim.auc_drop[::-1], color="#2a6f97"); ax.set_xlabel("AUC drop when shuffled (sealed month, post-hoc)")
plt.tight_layout(); plt.savefig("/content/fig3_importance.png", dpi=160); plt.close()


Base rate by label month: {'Jan label': np.float64(0.178), 'Feb label': np.float64(0.146), 'Mar label': np.float64(0.226), 'Apr label': np.float64(0.494), 'May label': np.float64(0.528), 'Jun label (sealed)': np.float64(0.613)}


In [25]:
import zipfile
PUBLIC = ["results_dev.csv","results_forward.csv","results_forward2.csv","results_sealed.csv","results_sealed.json",
          "sealed_per_client_base_rate.csv","perm_importance_modelB.csv","playbook_summary.csv","reason_code_summary.csv",
          "fig1_sealed_auc.png","fig2_base_rate_by_month.png","fig3_importance.png"]
with zipfile.ZipFile("/content/capstone_outputs.zip", "w") as z:
    for f in PUBLIC: z.write(f"/content/{f}", f)
print("Zipped public-safe outputs ->", "/content/capstone_outputs.zip", "(private ranked list NOT included)")

Zipped public-safe outputs -> /content/capstone_outputs.zip (private ranked list NOT included)


## ML-12: demo, social post, employer summary

**5-minute demo outline**
1. (0:30) The decision: which pages should an editor open first?
2. (1:00) The data and the label: 60-day features, 30-day label, sealed June test.
3. (1:00) The honest development story: near-null, rules flip sign, base rate swings.
4. (1:30) The sealed result: Model B AUC 0.640 vs 0.469 and 0.516, with the base-rate caveat.
5. (1:00) The action list: tiers, reason codes, what an editor does tomorrow.

**Social post cut (edit before posting)**
I built a page-level search-decline ranking on the FlyRank ML Internship dataset. Honest headline: in development it barely beat simple rules; on one sealed month it did (AUC 0.64 vs 0.47 and 0.52), in a month where 61% of pages declined. It is decision-support, not a forecast. Paper: [your paper link]

**Employer summary (3 sentences)**
I built and validated a page-ranking model on real search data, using client-grouped, time-forward and sealed-month evaluation. I reported a null development result and one modest sealed-month win with the limits stated, and turned the output into a tiered review list with reason codes. The work is deployed as a public research paper with a reproducible notebook.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
